<a href="https://colab.research.google.com/github/LUCASEPITACIO12/LUCASEPITACIO12/blob/main/script_plano_de_amostragem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Plano Anual (Rede/Distribuição)
from math import ceil
import re
import pandas as pd
import calendar

# Colab widgets
from google.colab import output
output.enable_custom_widget_manager()
import ipywidgets as widgets
from IPython.display import display, clear_output

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

MESES = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
         "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]

def ceil_div(a, b):
    return int(ceil(a / b))

def limpar_prefixo(nome: str) -> str:
    return re.sub(r"^\s*\d+\.\s*", "", nome or "").strip()

# ✅ NOVO: conta quantas vezes um dia da semana ocorre no mês
def ocorrencias_weekday_no_mes(ano: int, mes: int, weekday: int) -> int:
    """
    weekday: 0=Seg, 1=Ter, 2=Qua, 3=Qui, 4=Sex, 5=Sáb, 6=Dom
    """
    cal = calendar.monthcalendar(ano, mes)
    return sum(1 for semana in cal if semana[weekday] != 0)

# ===== Regras (Distribuição) =====
def anexo14_dist_mensal(pop: int) -> int:
    if pop < 5000: return 5
    if pop < 10000: return 10
    if pop <= 50000: return ceil_div(pop, 1000)
    if pop <= 80000: return 25 + ceil_div(pop, 2000)
    if pop <= 130000: return 1 + ceil_div(pop, 1250)
    if pop <= 250000: return 40 + ceil_div(pop, 2000)
    if pop <= 340000: return 115 + ceil_div(pop, 5000)
    if pop <= 400000: return 47 + ceil_div(pop, 2500)
    if pop <= 600000: return 127 + ceil_div(pop, 5000)
    if pop <= 1140000: return 187 + ceil_div(pop, 10000)
    return min(400, 244 + ceil_div(pop, 20000))

def psd_qtd_freq(manancial: str, pop: int):
    m = (manancial or "").strip().lower()
    if m == "superficial":
        if pop < 50000:   return (1, "Bimestral")
        if pop <= 250000: return (4, "Bimestral")
        return (8, "Bimestral")
    else:
        if pop < 50000:   return (1, "Anual")
        if pop <= 250000: return (2, "Semestral")
        return (3, "Semestral")

def meses_evento(freq: str):
    f = (freq or "").strip().lower()
    if f == "mensal": return list(range(1, 13))
    if f == "bimestral": return [1,3,5,7,9,11]
    if f == "trimestral": return [1,4,7,10]
    if f == "semestral": return [1,7]
    if f == "anual": return [1]
    # ⚠️ "semanal" não é tratado aqui porque depende do calendário e do weekday
    return []

# ✅ NOVO: calcula quanto cai no mês, tratando "Semanal" corretamente por calendário
def qtd_no_mes(freq: str, qtd: int, ano: int, mes_idx: int, weekday: int) -> int:
    f = (freq or "").strip().lower()

    if f == "mensal":
        return int(qtd)

    if f in ["bimestral", "trimestral", "semestral", "anual"]:
        return int(qtd) if mes_idx in meses_evento(freq) else 0

    if f == "semanal":
        # ✅ regra correta: nº de ocorrências do weekday no mês
        return int(qtd) * ocorrencias_weekday_no_mes(int(ano), int(mes_idx), int(weekday))

    # Se aparecer algo inesperado, não soma nada (evita erro silencioso)
    return 0

def grade_anual_distribuicao(
    pop: int,
    ano: int,
    manancial: str,
    usa_acrilamida: bool,
    usa_epicloridrina: bool,
    demais_status: str,  # "Detectado", "ND", "Não sei"
    incluir_linhas_dispensadas: bool,
    incluir_radioatividade_extra: bool,
    weekday_semanal: int,  # ✅ NOVO
):
    linhas = []

    qtd_mensal = anexo14_dist_mensal(pop)
    linhas.append((limpar_prefixo("1. Coliformes Totais, Cor aparente, pH, Turbidez, Residual e E. coli"),
                   qtd_mensal, "Mensal", True, "Obrigatório"))

    q_psd, f_psd = psd_qtd_freq(manancial, pop)
    linhas.append((limpar_prefixo("1. Produtos Secundários da Desinfecção (PSD)"),
                   q_psd, f_psd, True, "Obrigatório"))

    if demais_status == "ND":
        linhas.append((limpar_prefixo("1. Demais parâmetros"), 1, "Trimestral", False, "Dispensado (ND na saída)"))
    else:
        linhas.append((limpar_prefixo("1. Demais parâmetros"), 1, "Trimestral", True, "Obrigatório"))

    linhas.append((limpar_prefixo("1. Cloreto de Vinila"), 1, "Semestral", True, "Obrigatório"))

    linhas.append((limpar_prefixo("1. Acrilamida (se usa polímero)"), 1, "Mensal", usa_acrilamida, "Condicional"))
    linhas.append((limpar_prefixo("1. Epicloridrina (quando aplicável)"), 1, "Mensal", usa_epicloridrina, "Condicional"))

    if incluir_radioatividade_extra:
        linhas.append((limpar_prefixo("1. Radioatividade Alfa e Beta (EXTRA)"), 1, "Semestral", True, "Extra"))

    if incluir_linhas_dispensadas:
        linhas.append((limpar_prefixo("Fluoreto (dispensado na distribuição)"), 1, "Mensal", False, "Dispensado"))
        linhas.append((limpar_prefixo("Gosto e odor (dispensado na distribuição)"), 1, "Trimestral", False, "Dispensado"))
        # Mesmo dispensado, deixei "Semanal" suportado corretamente caso você ative depois
        linhas.append((limpar_prefixo("Cianotoxinas (dispensado na distribuição)"), 1, "Semanal", False, "Dispensado"))

    registros = []
    for (nome, qtd, freq, ativo, status) in linhas:
        row = {"Análise": nome, "Qtd": (qtd if ativo else "-"),
               "Frequência": (freq if ativo else "-"), "Status": status}

        total_row = 0
        for mes_idx, mes_nome in enumerate(MESES, start=1):
            if ativo:
                valor_mes = qtd_no_mes(freq=freq, qtd=qtd, ano=ano, mes_idx=mes_idx, weekday=weekday_semanal)
                if valor_mes > 0:
                    row[mes_nome] = int(valor_mes)
                    total_row += int(valor_mes)
                else:
                    row[mes_nome] = "-"
            else:
                row[mes_nome] = "-"

        row["Total anual"] = total_row if ativo else "-"
        registros.append(row)

    df = pd.DataFrame(registros, columns=["Análise","Qtd","Frequência","Status",*MESES,"Total anual"])
    return df

def mostrar_bonito(df: pd.DataFrame, titulo: str):
    sty = (df.style
           .set_properties(**{"border":"1px solid #333", "text-align":"center", "white-space":"pre-wrap"})
           .set_table_styles([
               {"selector":"th", "props":[("border","1px solid #333"),
                                         ("background","#e6e6e6"),
                                         ("text-align","center"),
                                         ("font-weight","bold")]},
               {"selector":"td", "props":[("border","1px solid #333")]}
           ])
           .hide(axis="index"))
    display(widgets.HTML(f"<h3 style='margin:0 0 8px 0;'>{titulo}</h3>"))
    display(sty)

def exportar_grade_excel(df: pd.DataFrame, ano: int, titulo: str, arquivo: str, pop: int, manancial: str, weekday_nome: str):
    wb = Workbook()
    ws = wb.active
    ws.title = "Distribuição"

    header_fill = PatternFill("solid", fgColor="BDD7EE")
    gray_fill   = PatternFill("solid", fgColor="E6E6E6")
    bold = Font(bold=True)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_wrap = Alignment(horizontal="left", vertical="top", wrap_text=True)

    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    ncols = len(df.columns)

    ws["A1"] = titulo
    ws["A1"].font = Font(bold=True, size=12)
    ws["A1"].alignment = center
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=ncols)
    ws["A1"].fill = header_fill

    ws["A2"] = f"Ano: {ano} | População: {pop} | Manancial: {manancial} | Dia semanal (p/ freq. semanal): {weekday_nome}"
    ws["A2"].alignment = center
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=ncols)
    ws["A2"].fill = gray_fill
    ws["A2"].font = Font(bold=True)

    for j, col in enumerate(df.columns, start=1):
        cell = ws.cell(row=3, column=j, value=col)
        cell.fill = header_fill
        cell.font = bold
        cell.alignment = center

    for i, row in enumerate(dataframe_to_rows(df, index=False, header=False), start=4):
        for j, val in enumerate(row, start=1):
            cell = ws.cell(row=i, column=j, value=val)
            cell.alignment = left_wrap if df.columns[j-1] == "Análise" else center

    for r in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in r:
            cell.border = border

    widths = {"Análise": 58, "Qtd": 8, "Frequência": 14, "Status": 20, "Total anual": 12}
    for j, col in enumerate(df.columns, start=1):
        w = widths.get(col, 14)
        ws.column_dimensions[ws.cell(row=3, column=j).column_letter].width = w

    ws.row_dimensions[1].height = 24
    ws.row_dimensions[2].height = 18
    ws.row_dimensions[3].height = 28
    for r in range(4, ws.max_row+1):
        ws.row_dimensions[r].height = 42

    wb.save(arquivo)
    return arquivo

# ===== UI =====
w_pop = widgets.IntText(description="População:", value=26000)
w_manancial = widgets.Dropdown(description="Manancial:", options=["Superficial", "Subterrâneo"], value="Superficial")
w_acri = widgets.ToggleButtons(description="Acrilamida?", options=["SIM","NÃO"], value="NÃO")
w_epic = widgets.ToggleButtons(description="Epicloridrina?", options=["SIM","NÃO"], value="NÃO")
w_demais = widgets.Dropdown(description="Demais (saída):", options=["Detectado", "ND", "Não sei"], value="Não sei")
w_disp = widgets.ToggleButtons(description="Mostrar dispensados?", options=["SIM","NÃO"], value="NÃO")
w_radio = widgets.ToggleButtons(description="Radioatividade EXTRA?", options=["SIM","NÃO"], value="NÃO")
w_ano = widgets.IntText(description="Ano:", value=2026)

# ✅ NOVO: dia da semana para frequência SEMANAL (evita “fazer a mais”)
weekday_opts = [("Segunda",0),("Terça",1),("Quarta",2),("Quinta",3),("Sexta",4),("Sábado",5),("Domingo",6)]
w_weekday = widgets.Dropdown(description="Dia semanal:", options=weekday_opts, value=3)

btn = widgets.Button(description="Gerar grade anual", button_style="success")
btn_xlsx = widgets.Button(description="Baixar Excel", button_style="info")

out = widgets.Output()
_last = {"df": None}

def on_gerar(_):
    with out:
        clear_output()
        df = grade_anual_distribuicao(
            pop=int(w_pop.value),
            ano=int(w_ano.value),
            manancial=w_manancial.value,
            usa_acrilamida=(w_acri.value == "SIM"),
            usa_epicloridrina=(w_epic.value == "SIM"),
            demais_status=w_demais.value,
            incluir_linhas_dispensadas=(w_disp.value == "SIM"),
            incluir_radioatividade_extra=(w_radio.value == "SIM"),
            weekday_semanal=int(w_weekday.value),  # ✅ NOVO
        )
        _last["df"] = df
        display(widgets.HTML(
            "<div style='font-size:12px;color:#444;margin:0 0 8px 0;'>"
            "Obs.: Se existir alguma análise com frequência <b>Semanal</b>, o mês recebe "
            "<b>(nº de ocorrências do dia semanal escolhido)</b> × Qtd, evitando dupla contagem na virada do mês "
            "e fechando 52/53 no ano."
            "</div>"
        ))
        mostrar_bonito(df, "Plano Anual – Rede/Distribuição (grade mensal + total anual por análise)")

def on_baixar(_):
    if _last["df"] is None or _last["df"].empty:
        with out:
            print("Gere a grade primeiro.")
        return
    nome = f"Plano_Anual_Distribuicao_{w_ano.value}.xlsx"
    weekday_nome = dict(weekday_opts).get(int(w_weekday.value), str(w_weekday.value))
    exportar_grade_excel(
        _last["df"],
        int(w_ano.value),
        "Plano Anual – Rede/Distribuição",
        nome,
        pop=int(w_pop.value),
        manancial=w_manancial.value,
        weekday_nome=weekday_nome
    )
    from google.colab import files
    files.download(nome)

btn.on_click(on_gerar)
btn_xlsx.on_click(on_baixar)

ui = widgets.VBox([
    widgets.HTML("<h3>Plano Anual (Rede/Distribuição) – organizado</h3>"),
    widgets.HBox([w_ano, w_pop, w_manancial]),
    widgets.HBox([w_weekday]),  # ✅ NOVO
    widgets.HBox([w_acri, w_epic]),
    widgets.HBox([w_demais]),
    widgets.HBox([w_disp, w_radio]),
    widgets.HBox([btn, btn_xlsx]),
    out
])

display(ui)


In [ ]:
# @title PLANO DE AMOSTRAGEM (ÁGUA BRUTA)

import calendar
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import output
output.enable_custom_widget_manager()

MESES_ABR = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
MESES_NUM = list(range(1, 13))

FREQ_ANO = {
    "ANUAL": 1,
    "SEMESTRAL": 2,
    "TRIMESTRAL": 4,
    "MENSAL": 12,
    "SEMANAL": 52,
}

# ---------- helpers ----------
def ocorrencias_semanais_no_mes(ano: int, mes: int, weekday: int) -> int:
    cal = calendar.monthcalendar(ano, mes)
    return sum(1 for semana in cal if semana[weekday] != 0)

def na_se_zero_pontos(n_pontos: int, valor: int):
    return "NA" if n_pontos == 0 else int(valor)

def soma_na(a, b):
    if a == "NA" and b == "NA": return "NA"
    return int((0 if a == "NA" else a) + (0 if b == "NA" else b))

def estilo(df: pd.DataFrame):
    return (df.style
            .hide(axis="index")
            .set_properties(**{"border":"1px solid #333", "text-align":"center", "white-space":"pre-wrap"})
            .set_table_styles([
                {"selector":"th", "props":[("border","1px solid #333"),
                                          ("background","#e6e6e6"),
                                          ("text-align","center"),
                                          ("font-weight","bold")]},
                {"selector":"td", "props":[("border","1px solid #333")]}
            ]))

def meses_semestre(inicio: int):
    return (inicio, ((inicio - 1 + 6) % 12) + 1)

def meses_trimestre(inicio: int):
    return tuple((((inicio - 1 + k) % 12) + 1) for k in (0, 3, 6, 9))

def mensal_por_freq(ano, freq, n_pontos, mes_ini_sem, mes_ini_tri, weekday):
    if n_pontos == 0:
        return [0]*12

    freq = freq.upper()
    if freq == "MENSAL":
        return [1*n_pontos]*12

    if freq == "TRIMESTRAL":
        arr = [0]*12
        for m in meses_trimestre(mes_ini_tri):
            arr[m-1] = 1*n_pontos
        return arr

    if freq == "SEMESTRAL":
        arr = [0]*12
        for m in meses_semestre(mes_ini_sem):
            arr[m-1] = 1*n_pontos
        return arr

    if freq == "ANUAL":
        arr = [0]*12
        arr[0] = 1*n_pontos
        return arr

    if freq == "SEMANAL":
        return [ocorrencias_semanais_no_mes(ano, mes, weekday)*n_pontos for mes in MESES_NUM]

    raise ValueError("Frequência inválida.")

# ---------- Tabela 1: Pontos de Captação (anual) ----------
def tabela_pontos_captacao(n_sup, n_sub, ciano_cat, alt_cianotox, gat_esporos, gat_giardia, gat_cianotox):
    """
    IMPORTANTE:
    - 'Pontos subterrâneos' aqui deve representar QUANTOS pontos existem
      onde é possível coletar ÁGUA BRUTA antes da desinfecção (ex.: linha do poço antes do cloro).
    """
    linhas = []
    freq_ciano = "SEMANAL" if ciano_cat == ">10000" else "TRIMESTRAL"

    if not alt_cianotox:
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO[freq_ciano]); sub = "NA"
        linhas.append(("Cianobactérias", sub, sup, soma_na(sub, sup)))

        # ✅ ATUALIZADO (E. coli subterrânea): mensal em captação subterrânea (antes da desinfecção)
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["MENSAL"])
        sub = na_se_zero_pontos(n_sub, n_sub * FREQ_ANO["MENSAL"])
        linhas.append(("Escherichia coli", sub, sup, soma_na(sub, sup)))

        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["MENSAL"]); sub = "NA"
        linhas.append(("Clorofila-a", sub, sup, soma_na(sub, sup)))
    else:
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["SEMANAL"]); sub = "NA"
        linhas.append(("Cianotoxinas (alternativa: semanal na água bruta)", sub, sup, soma_na(sub, sup)))

        # ✅ ATUALIZADO (E. coli subterrânea): mensal em captação subterrânea (antes da desinfecção)
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["MENSAL"])
        sub = na_se_zero_pontos(n_sub, n_sub * FREQ_ANO["MENSAL"])
        linhas.append(("Escherichia coli", sub, sup, soma_na(sub, sup)))

    sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["SEMESTRAL"]); sub = "NA"
    linhas.append(("DQO, DBO e OD", sub, sup, soma_na(sub, sup)))

    sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["SEMESTRAL"])
    sub = na_se_zero_pontos(n_sub, n_sub * FREQ_ANO["SEMESTRAL"])
    linhas.append(("Turbidez, Cor Verdadeira, pH, Fósforo Total e Nitrogênio Amoniacal Total", sub, sup, soma_na(sub, sup)))

    sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["SEMESTRAL"])
    sub = na_se_zero_pontos(n_sub, n_sub * FREQ_ANO["SEMESTRAL"])
    linhas.append(("Substâncias químicas inorgânicas, orgânicas e agrotóxicos", sub, sup, soma_na(sub, sup)))

    if gat_esporos:
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["SEMANAL"]); sub = "NA"
        linhas.append(("Esporos de bactérias aeróbias (gatilho)", sub, sup, soma_na(sub, sup)))

    if gat_giardia:
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["MENSAL"]); sub = "NA"
        linhas.append(("Giardia/Crypto (gatilho)", sub, sup, soma_na(sub, sup)))

    if gat_cianotox and (not alt_cianotox):
        sup = na_se_zero_pontos(n_sup, n_sup * FREQ_ANO["SEMANAL"]); sub = "NA"
        linhas.append(("Cianotoxinas (gatilho)", sub, sup, soma_na(sub, sup)))

    return pd.DataFrame(linhas, columns=[
        "Parâmetros",
        "Total anual de análises em pontos\nde captação subterrânea",
        "Total anual de análises em pontos\nde captação superficial",
        "Total anual\nde análises",
    ])

# ---------- Tabela 2: Controle mensal ----------
def tabela_controle_mensal(ano, n_sup, n_sub, ciano_cat, alt_cianotox, gat_esporos, gat_giardia, gat_cianotox,
                           mes_ini_sem, mes_ini_tri, weekday):
    linhas = []

    def add(nome, freq, sup=True, sub=False):
        sup_arr = mensal_por_freq(ano, freq, (n_sup if sup else 0), mes_ini_sem, mes_ini_tri, weekday)
        sub_arr = mensal_por_freq(ano, freq, (n_sub if sub else 0), mes_ini_sem, mes_ini_tri, weekday)
        total = [a+b for a, b in zip(sup_arr, sub_arr)]
        linhas.append([nome] + total + [sum(total)])

    freq_ciano = "SEMANAL" if ciano_cat == ">10000" else "TRIMESTRAL"

    if not alt_cianotox:
        add("Cianobactérias (superficial)", freq_ciano, sup=True, sub=False)

        # ✅ ATUALIZADO (E. coli subterrânea): agora soma superficial + subterrânea mensal
        add("Escherichia coli (captação – sup+sub)", "MENSAL", sup=True, sub=True)

        add("Clorofila-a (superficial)", "MENSAL", sup=True, sub=False)
    else:
        add("Cianotoxinas (alternativa – superficial)", "SEMANAL", sup=True, sub=False)

        # ✅ ATUALIZADO (E. coli subterrânea): agora soma superficial + subterrânea mensal
        add("Escherichia coli (captação – sup+sub)", "MENSAL", sup=True, sub=True)

    add("DQO/DBO/OD (superficial)", "SEMESTRAL", sup=True, sub=False)
    add("Turbidez/Cor/pH/Fósforo/NH3-total (sup+sub)", "SEMESTRAL", sup=True, sub=True)
    add("Inorgânicos/Orgânicos/Agrotóxicos (sup+sub)", "SEMESTRAL", sup=True, sub=True)

    if gat_esporos:
        add("Esporos aeróbios (gatilho – superficial)", "SEMANAL", sup=True, sub=False)
    if gat_giardia:
        add("Giardia/Crypto (gatilho – superficial)", "MENSAL", sup=True, sub=False)
    if gat_cianotox and (not alt_cianotox):
        add("Cianotoxinas (gatilho – superficial)", "SEMANAL", sup=True, sub=False)

    cols = ["Parâmetros"] + MESES_ABR + ["Total ano"]
    return pd.DataFrame(linhas, columns=cols)

# =========================
# UI (ENXUTA)
# =========================
w_ano = widgets.IntText(value=2026, description="Ano:", layout=widgets.Layout(width="160px"))
w_sup = widgets.BoundedIntText(value=1, min=0, max=999, description="Pontos superficiais:")
w_sub = widgets.BoundedIntText(value=0, min=0, max=999, description="Pontos subterrâneos:")

w_ciano = widgets.Dropdown(
    options=[("≤ 10.000 cél/mL (trimestral)", "<=10000"),
             ("> 10.000 cél/mL (semanal)", ">10000")],
    value="<=10000",
    description="Cianobact.:"
)
w_alt = widgets.Checkbox(value=False, description="Alternativa: CIANOTOXINAS semanal (dispensa cianobactérias + clorofila-a)")

w_gat_esporos = widgets.Checkbox(value=False, description="Gatilho: Esporos aeróbios")
w_gat_giardia = widgets.Checkbox(value=False, description="Gatilho: Giardia/Crypto")
w_gat_cianotox = widgets.Checkbox(value=False, description="Gatilho: Cianotoxinas")

w_ini_sem = widgets.Dropdown(
    options=[(m, i) for i, m in enumerate(MESES_ABR, start=1)],
    value=1,
    description="Início semestral:"
)
w_ini_tri = widgets.Dropdown(
    options=[(m, i) for i, m in enumerate(MESES_ABR, start=1)],
    value=1,
    description="Início trimestral:"
)
w_week = widgets.Dropdown(
    options=[("Segunda",0),("Terça",1),("Quarta",2),("Quinta",3),("Sexta",4),("Sábado",5),("Domingo",6)],
    value=0,
    description="Semanal:"
)

btn = widgets.Button(description="Gerar / Atualizar", button_style="success", icon="refresh")
out = widgets.Output()

def render(_=None):
    with out:
        clear_output(wait=True)

        display(widgets.HTML(
            f"<h3 style='margin:0 0 6px 0;'>Plano de Amostragem — Água Bruta</h3>"
            f"<div style='margin:0 0 10px 0;'>"
            f"<b>Ano:</b> {w_ano.value} &nbsp; | &nbsp; "
            f"<b>Pontos:</b> Sup={w_sup.value}, Sub={w_sub.value} &nbsp; | &nbsp; "
            f"<b>Início:</b> Semestral={MESES_ABR[w_ini_sem.value-1]} / Trimestral={MESES_ABR[w_ini_tri.value-1]}"
            f"</div>"
            f"<div style='margin:0 0 10px 0; font-size:12px; color:#444;'>"
            f"Obs.: 'Pontos subterrâneos' = nº de pontos onde se coleta água bruta <b>antes da desinfecção</b> (E. coli mensal)."
            f"</div>"
        ))

        df1 = tabela_pontos_captacao(
            n_sup=w_sup.value,
            n_sub=w_sub.value,
            ciano_cat=w_ciano.value,
            alt_cianotox=w_alt.value,
            gat_esporos=w_gat_esporos.value,
            gat_giardia=w_gat_giardia.value,
            gat_cianotox=w_gat_cianotox.value
        )
        display(widgets.HTML("<h4 style='margin:8px 0 6px 0;'>Pontos de Captação</h4>"))
        display(estilo(df1))

        df2 = tabela_controle_mensal(
            ano=w_ano.value,
            n_sup=w_sup.value,
            n_sub=w_sub.value,
            ciano_cat=w_ciano.value,
            alt_cianotox=w_alt.value,
            gat_esporos=w_gat_esporos.value,
            gat_giardia=w_gat_giardia.value,
            gat_cianotox=w_gat_cianotox.value,
            mes_ini_sem=w_ini_sem.value,
            mes_ini_tri=w_ini_tri.value,
            weekday=w_week.value
        )
        display(widgets.HTML("<h4 style='margin:12px 0 6px 0;'>Controle mensal (Jan–Dez) — total já somando todos os pontos</h4>"))
        display(estilo(df2))

btn.on_click(render)

ui = widgets.VBox([
    widgets.HBox([w_ano, w_sup, w_sub]),
    widgets.HTML("<hr><b>Regras específicas</b>"),
    w_ciano,
    w_alt,
    widgets.HTML("<hr><b>Gatilhos (marque só se aplicar ao sistema)</b>"),
    w_gat_esporos, w_gat_giardia, w_gat_cianotox,
    widgets.HTML("<hr><b>Preferência do calendário (simplificado)</b>"),
    widgets.HBox([w_ini_sem, w_ini_tri, w_week]),
    btn,
    out
])

display(ui)
render()




In [ ]:
# @title Pós-filtração (Turbidez)
# Controle mês a mês (dias reais do ano) — Pós-filtração (Turbidez) — SOMENTE ESSA TABELA
# Colab/Jupyter com seleção (checkbox/slider/dropdown) — sem precisar escrever regra

import math
import pandas as pd
import calendar
from IPython.display import display, clear_output

# Se estiver no Google Colab e o painel NÃO aparecer, descomente:
# !pip -q install ipywidgets
# from google.colab import output; output.enable_custom_widget_manager()

import ipywidgets as widgets


# -----------------------------
# Regra: Turbidez pós-filtração
# -----------------------------
# 1 amostra por evento
# - Filtração rápida / membrana: 1 a cada 2 horas
# - Filtração lenta: 1 diária

def calc_mensal_por_filtro(tipo_tratamento: str, horas_operacao_dia: int, dias_mes: int) -> int:
    if tipo_tratamento in ["Filtração rápida/membrana (1 a cada 2h)"]:
        por_dia = math.ceil(horas_operacao_dia / 2)
        return int(por_dia * dias_mes)
    elif tipo_tratamento == "Filtração lenta (1/dia)":
        return int(dias_mes)
    else:
        raise ValueError("Tipo de tratamento inválido.")

def montar_tabela_mensal_controle(ano, tipo_tratamento, n_filtros, horas_operacao_dia):
    meses = [
        (1, "Jan"), (2, "Fev"), (3, "Mar"), (4, "Abr"),
        (5, "Mai"), (6, "Jun"), (7, "Jul"), (8, "Ago"),
        (9, "Set"), (10, "Out"), (11, "Nov"), (12, "Dez")
    ]

    linhas = []
    for m, nome in meses:
        dias = calendar.monthrange(int(ano), m)[1]
        mensal_por_filtro = calc_mensal_por_filtro(tipo_tratamento, horas_operacao_dia, dias)
        total_mensal = mensal_por_filtro * n_filtros
        linhas.append({
            "Mês": nome,
            "Dias": dias,
            "Análises/mês por filtro": mensal_por_filtro,
            "Total mensal (todos os filtros)": total_mensal
        })

    df = pd.DataFrame(linhas)
    total_anual = int(df["Total mensal (todos os filtros)"].sum())
    return df, total_anual


# -----------------------------
# UI (clicável)
# -----------------------------
w_ano = widgets.IntText(value=2026, description="Ano:", layout=widgets.Layout(width="200px"))

w_tipo = widgets.Dropdown(
    options=[
        "Filtração rápida/membrana (1 a cada 2h)",
        "Filtração lenta (1/dia)"
    ],
    value="Filtração rápida/membrana (1 a cada 2h)",
    description="Tratamento:",
    layout=widgets.Layout(width="520px")
)

w_filtros = widgets.IntSlider(
    value=2, min=1, max=30, step=1,
    description="Nº filtros:",
    continuous_update=False,
    layout=widgets.Layout(width="520px")
)

w_horas = widgets.IntSlider(
    value=24, min=1, max=24, step=1,
    description="Horas/dia:",
    continuous_update=False,
    layout=widgets.Layout(width="520px")
)

out = widgets.Output()

def _habilitar_campos(*args):
    w_horas.disabled = (w_tipo.value == "Filtração lenta (1/dia)")

def _style_table(df):
    return (df.style
            .set_properties(**{"border": "1px solid #333", "text-align": "center"})
            .set_table_styles([
                {"selector":"th", "props":[("border","1px solid #333"),
                                          ("background","#e6e6e6"),
                                          ("text-align","center")]},
                {"selector":"td", "props":[("border","1px solid #333")]}
            ])
            .hide(axis="index"))

def render(*args):
    with out:
        clear_output(wait=True)

        df_meses, total_anual = montar_tabela_mensal_controle(
            ano=int(w_ano.value),
            tipo_tratamento=w_tipo.value,
            n_filtros=int(w_filtros.value),
            horas_operacao_dia=int(w_horas.value)
        )

        display(widgets.HTML("<h3 style='margin:0 0 6px 0;'>Controle mês a mês (dias reais do ano)</h3>"))
        display(_style_table(df_meses))

        display(widgets.HTML(f"<b>Total anual (somatório dos meses):</b> {total_anual}"))

        obs = (
            "(1) Turbidez pós-filtração: 1 amostra por evento.\n\n"
            "• Filtração rápida/membrana: 1 a cada 2h (cálculo usa teto(horas/2) por dia).\n"
            "• Filtração lenta: 1 amostra por dia."
        )
        display(widgets.HTML(f"<small><pre style='margin:10px 0 0 0; white-space:pre-wrap;'>{obs}</pre></small>"))

for w in [w_ano, w_tipo, w_filtros, w_horas]:
    w.observe(render, names="value")
w_tipo.observe(_habilitar_campos, names="value")

ui = widgets.VBox([w_ano, w_tipo, w_filtros, w_horas])
display(ui, out)

_habilitar_campos()
render()


Output()

In [ ]:
# @title Saída do Tratamento — Controle mensal
import math
import pandas as pd
import calendar
from IPython.display import display, clear_output
import ipywidgets as widgets

# Se estiver no Google Colab e o painel NÃO aparecer, descomente:
# !pip -q install ipywidgets
# from google.colab import output; output.enable_custom_widget_manager()

MESES = [
    (1, "Jan"), (2, "Fev"), (3, "Mar"), (4, "Abr"),
    (5, "Mai"), (6, "Jun"), (7, "Jul"), (8, "Ago"),
    (9, "Set"), (10, "Out"), (11, "Nov"), (12, "Dez")
]

def _style_table(df, hide_index=True):
    sty = (df.style
           .set_properties(**{"border": "1px solid #333", "text-align": "center", "white-space": "pre-wrap"})
           .set_table_styles([
               {"selector":"th", "props":[("border","1px solid #333"),
                                         ("background","#e6e6e6"),
                                         ("text-align","center")]},
               {"selector":"td", "props":[("border","1px solid #333")]}
           ]))
    if hide_index:
        sty = sty.hide(axis="index")
    return sty

def _meses_trimestral(mes_inicio):
    base = int(mes_inicio)
    return [((base - 1 + k) % 12) + 1 for k in [0, 3, 6, 9]]

def _meses_semestral(mes_inicio):
    base = int(mes_inicio)
    return [base, ((base - 1 + 6) % 12) + 1]

# ✅ NOVO: conta quantas vezes um dia da semana ocorre no mês (evita dupla contagem na virada do mês)
def ocorrencias_weekday_no_mes(ano: int, mes: int, weekday: int) -> int:
    """
    weekday: 0=Seg, 1=Ter, 2=Qua, 3=Qui, 4=Sex, 5=Sáb, 6=Dom
    """
    cal = calendar.monthcalendar(ano, mes)
    return sum(1 for semana in cal if semana[weekday] != 0)

# ✅ ATUALIZADO: agora recebe ano e weekday para calcular semanal correto
def _eventos_mensais(freq, metodo_mensal, horas_dia, dias_mes, mes_num, meses_tri, meses_sem, ano, weekday):
    if freq == "NA":
        return None

    # Mantém compatibilidade com o "modelo planilha"
    if metodo_mensal == "Modelo planilha (30 dias; 4 semanas)":
        dias_mes = 30

    if freq == "A cada 2 horas":
        por_dia = math.ceil(int(horas_dia) / 2)
        return por_dia * int(dias_mes)

    if freq == "Diária":
        return int(dias_mes)

    if freq == "Semanal":
        # Modelo planilha fixa 4 semanas por mês
        if metodo_mensal == "Modelo planilha (30 dias; 4 semanas)":
            return 4
        # ✅ Correto para dias reais: conta ocorrências do weekday no mês
        return ocorrencias_weekday_no_mes(int(ano), int(mes_num), int(weekday))

    if freq == "Mensal":
        return 1

    if freq == "Trimestral":
        return 1 if int(mes_num) in meses_tri else 0

    if freq == "Semestral":
        return 1 if int(mes_num) in meses_sem else 0

    raise ValueError("Frequência inválida.")

def regras_saida_tratamento(manancial, fluoretacao, ciano_gt_20000, acrilamida, epicloridrina):
    manancial = manancial.lower().strip()

    # Regras por tipo de manancial (como nas imagens)
    if manancial == "superficial":
        freq_cor_turb_res = "A cada 2 horas"
        freq_ph = "A cada 2 horas"
        coliformes_n = 2
        coliformes_freq = "Semanal"
        gosto_odor_freq = "Trimestral"
    else:
        freq_cor_turb_res = "Semanal"
        freq_ph = "Semanal"
        coliformes_n = 1
        coliformes_freq = "Semanal"
        gosto_odor_freq = "Semestral"

    linhas = [
        ("Coliformes totais", coliformes_n, coliformes_freq),
        ("Cor aparente,\nTurbidez, Residual de\ndesinfetante", 1, freq_cor_turb_res),
        ("pH", 1, freq_ph),
        ("Gosto\ne odor", 1, gosto_odor_freq),
    ]

    # Cianotoxinas (gatilho)
    if manancial == "superficial" and ciano_gt_20000:
        linhas.append(("Cianotoxinas", 1, "Semanal"))
    else:
        linhas.append(("Cianotoxinas", "NA", "NA"))

    # Fluoreto (se houver fluoretação/desfluoretação)
    if fluoretacao:
        linhas.append(("Fluoreto", 1, "A cada 2 horas"))
    else:
        linhas.append(("Fluoreto", "NA", "NA"))

    # Acrilamida/Epicloridrina (somente se usar polímero correspondente)
    linhas.append(("Acrilamida", 1, "Mensal") if acrilamida else ("Acrilamida", "NA", "NA"))
    linhas.append(("Epicloridrina", 1, "Mensal") if epicloridrina else ("Epicloridrina", "NA", "NA"))

    # Fixos
    linhas.append(("Cloreto\nde Vinila", 1, "Semestral"))
    linhas.append(("Demais parâmetros\n(ANEXO IX e XI)", 1, "Semestral"))

    return linhas


# =========================
# UI (clicável)
# =========================
w_ano = widgets.IntSlider(value=2026, min=2020, max=2035, step=1, description="Ano:", continuous_update=False)
w_manancial = widgets.Dropdown(
    options=["Superficial", "Subterrâneo"],
    value="Superficial",
    description="Manancial:",
    layout=widgets.Layout(width="260px")
)
w_horas = widgets.IntSlider(value=24, min=1, max=24, step=1, description="Horas/dia:", continuous_update=False)

w_metodo_mensal = widgets.Dropdown(
    options=["Dias reais do mês", "Modelo planilha (30 dias; 4 semanas)"],
    value="Dias reais do mês",
    description="Cálculo mensal:",
    layout=widgets.Layout(width="520px")
)

w_ini_tri = widgets.Dropdown(
    options=[(n, i) for i, n in MESES],
    value=1,
    description="Início trimestral:",
    layout=widgets.Layout(width="260px")
)
w_ini_sem = widgets.Dropdown(
    options=[(n, i) for i, n in MESES],
    value=1,
    description="Início semestral:",
    layout=widgets.Layout(width="260px")
)

# ✅ NOVO: ancora a regra semanal em um dia fixo da semana (para dar 52/53 no ano, sem dupla contagem)
w_weekday = widgets.Dropdown(
    options=[("Segunda",0),("Terça",1),("Quarta",2),("Quinta",3),("Sexta",4),("Sábado",5),("Domingo",6)],
    value=3,  # Quinta por padrão
    description="Dia semanal:",
    layout=widgets.Layout(width="260px")
)

w_fluor = widgets.Checkbox(value=False, description="Fluoretação/desfluoretação?")
w_ciano = widgets.Checkbox(value=False, description="Cianobactérias > 20.000 cél/mL?")
w_acri = widgets.Checkbox(value=False, description="Usa polímero com acrilamida?")
w_epi  = widgets.Checkbox(value=False, description="Usa polímero com epicloridrina?")

out = widgets.Output()

def render(*args):
    with out:
        clear_output(wait=True)

        ano = int(w_ano.value)
        horas = int(w_horas.value)
        meses_tri = _meses_trimestral(w_ini_tri.value)
        meses_sem = _meses_semestral(w_ini_sem.value)
        weekday = int(w_weekday.value)

        regras = regras_saida_tratamento(
            manancial=w_manancial.value,
            fluoretacao=bool(w_fluor.value),
            ciano_gt_20000=bool(w_ciano.value),
            acrilamida=bool(w_acri.value),
            epicloridrina=bool(w_epi.value),
        )

        linhas = []
        for mes_num, mes_nome in MESES:
            dias_mes = calendar.monthrange(ano, mes_num)[1]
            linha = {"Mês": mes_nome, "Dias": dias_mes}

            for param, n, freq in regras:
                if n == "NA" or freq == "NA":
                    linha[param] = "NA"
                else:
                    ev = _eventos_mensais(
                        freq=freq,
                        metodo_mensal=w_metodo_mensal.value,
                        horas_dia=horas,
                        dias_mes=dias_mes,
                        mes_num=mes_num,
                        meses_tri=meses_tri,
                        meses_sem=meses_sem,
                        ano=ano,
                        weekday=weekday
                    )
                    linha[param] = int(n) * int(ev)

            linhas.append(linha)

        df = pd.DataFrame(linhas)

        # Linha TOTAL (soma por parâmetro)
        totais = {"Mês": "Total", "Dias": ""}
        for col in df.columns:
            if col in ["Mês", "Dias"]:
                continue
            vals = pd.to_numeric(df[col], errors="coerce")  # NA -> NaN
            totais[col] = int(vals.fillna(0).sum())

        df_total = pd.concat([df, pd.DataFrame([totais])], ignore_index=True)

        display(widgets.HTML("<h3 style='margin:0 0 6px 0;'>Controle mensal (Jan–Dez) — Saída do Tratamento</h3>"))
        display(widgets.HTML(
            "<div style='font-size:12px;color:#444;margin:0 0 10px 0;'>"
            "Obs.: Para frequência <b>Semanal</b> no modo <b>Dias reais do mês</b>, o cálculo usa o nº de ocorrências do "
            "<b>dia semanal</b> escolhido no calendário (evita dupla contagem na virada do mês e fecha em 52/53 no ano)."
            "</div>"
        ))
        display(_style_table(df_total))

        total_geral = sum(
            int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())
            for col in df.columns if col not in ["Mês", "Dias"]
        )
        display(widgets.HTML(f"<b>Total anual (soma de todos os parâmetros):</b> {total_geral}"))

# Observers
for w in [w_ano, w_manancial, w_horas, w_metodo_mensal, w_ini_tri, w_ini_sem, w_weekday, w_fluor, w_ciano, w_acri, w_epi]:
    w.observe(render, names="value")

ui = widgets.VBox([
    widgets.HBox([w_ano, w_manancial]),
    w_horas,
    w_metodo_mensal,
    widgets.HBox([w_ini_tri, w_ini_sem]),
    w_weekday,
    widgets.HTML("<hr style='margin:10px 0;'>"),
    widgets.VBox([w_fluor, w_ciano, w_acri, w_epi]),
])

display(ui, out)
render()



Output()